In [1]:
#================================================================
# CELL 1: REPRODUCIBILITY + CONFIG
#================================================================
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f'All seeds fixed to {SEED}')

CFG = {
    # === DATA PATHS ===
    'cotton_path': '/kaggle/input/datasets/jawadulkarim117/cotton-weed-12-class',
    'mhweed_path': '/kaggle/input/datasets/sayalis069/mh-weed16/MH-Weed16An Indian Multiclass Annotated Weed Dataset for Computer Vision Tasks/MH-Weed16',
    'work_dir':    '/kaggle/working',

    # === MODEL ===
    'teacher_weights': '/kaggle/input/datasets/rahu12345/teacher/best.pt',  # ← UPDATED
    'student_weights': 'yolov8n',
    'nc': 28,

    # === IMAGE/BATCH ===
    'img_size':    640,
    'batch_size':  16,
    'num_workers': 2,

    # === TEACHER (already done) ===
    'teacher_epochs': 30,
    'teacher_lr':     1e-4,

    # === KD STUDENT TRAINING ===
    'kd_epochs':    120,
    'kd_lr':        1e-3,
    'weight_decay': 1e-4,
    'temperature':  3.0,
    'alpha':       0.5,   # detection loss
    'beta':        0.3,   # feature loss
    'gamma':       0.2,   # KL soft label loss

    # === MISC ===
    'msad_threshold': 0.75,
    'low_ap_warn':    0.50,
    'check_every':    10,
    'patience':       30,
    'save_every':     10,
}

assert abs(CFG['alpha'] + CFG['beta'] + CFG['gamma'] - 1.0) < 1e-6, 'alpha+beta+gamma must = 1.0'

from pathlib import Path
import yaml

OUTPUT    = Path(CFG['work_dir']) / 'outputs'
COTTON    = Path(CFG['cotton_path']) / 'cotton_weed'
MHWEED    = Path(CFG['mhweed_path'])
MERGED    = Path(CFG['work_dir']) / 'merged'
YAML_PATH = MERGED / 'data.yaml'

OUTPUT.mkdir(parents=True, exist_ok=True)

assert COTTON.exists(),                           f'Cotton not found: {COTTON}'
assert MHWEED.exists(),                           f'MHWeed not found: {MHWEED}'
assert Path(CFG['teacher_weights']).exists(),     f'Teacher weights not found!'

yaml.dump(CFG, open(OUTPUT / 'config.yaml', 'w'))
print(f'COTTON:          {COTTON}')
print(f'MHWEED:          {MHWEED}')
print(f'MERGED:          {MERGED}')
print(f'YAML_PATH:       {YAML_PATH}')
print(f'TEACHER WEIGHTS: ✅ {CFG["teacher_weights"]}')
if YAML_PATH.exists():
    print('YAML_PATH: ✅ exists (Cell 5 already run)')
else:
    print('YAML_PATH: ⚠ not yet created — run Cell 5 first')

All seeds fixed to 42
COTTON:          /kaggle/input/datasets/jawadulkarim117/cotton-weed-12-class/cotton_weed
MHWEED:          /kaggle/input/datasets/sayalis069/mh-weed16/MH-Weed16An Indian Multiclass Annotated Weed Dataset for Computer Vision Tasks/MH-Weed16
MERGED:          /kaggle/working/merged
YAML_PATH:       /kaggle/working/merged/data.yaml
TEACHER WEIGHTS: ✅ /kaggle/input/datasets/rahu12345/teacher/best.pt
YAML_PATH: ⚠ not yet created — run Cell 5 first


In [2]:
# ================================================================
# CELL 2 (STABLE): IMPORTS + GLOBALS
# ================================================================
# If imports fail, THEN install minimal packages.
# Prefer using Kaggle preinstalled env to avoid numpy/scipy breakage.

import os, re, gc, math, json, time, random, shutil, logging
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Try import first (no pip by default)
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except Exception:
    !pip -q install --no-deps albumentations==1.4.18
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

try:
    import ultralytics
    from ultralytics import YOLO
except Exception:
    !pip -q install --no-deps ultralytics==8.4.21
    import ultralytics
    from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("ultralytics:", ultralytics.__version__)
print("albumentations:", A.__version__)

# deterministic-ish
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

WORK   = Path(CFG['work_dir'])
OUTPUT = WORK / "outputs"
MERGED = WORK / "merged"
OUTPUT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler(OUTPUT / 'run.log')]
)
log = logging.getLogger("KD")
log.info("Bootstrap complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


2026-03-16 18:17:40,232 | INFO | Bootstrap complete.


Device: cuda
numpy: 2.0.2
torch: 2.9.0+cu126
ultralytics: 8.4.21
albumentations: 2.0.8


In [3]:
# ================================================================
# CELL 3 (REWRITE): CLASS DISCOVERY + UNIFICATION
# ================================================================
import re
from pathlib import Path

def clean_name(name: str) -> str:
    name = re.sub(r'^\d+\.+', '', name).strip()
    if '(' in name:
        name = name.split('(')[0].strip()
    name = name.replace('_', ' ')
    name = re.sub(r'\s+', ' ', name).strip().rstrip('.')
    return name.title()

# Cotton classes
cotton_yaml_path = Path(CFG['cotton_path']) / "data.yaml"
assert cotton_yaml_path.exists(), f"Missing cotton data.yaml: {cotton_yaml_path}"

with open(cotton_yaml_path, "r") as f:
    c_yaml = yaml.safe_load(f)

cotton_classes = c_yaml["names"]
if isinstance(cotton_classes, dict):
    cotton_classes = [cotton_classes[i] for i in range(len(cotton_classes))]
cotton_classes = [str(x).strip() for x in cotton_classes]

# MH-Weed classes from folders
mh_root = Path(CFG['mhweed_path'])
class_folder = mh_root / "Individual Weed Species" / "16 Classes of Weed_Species" / "Individual Weed_Species"
assert class_folder.exists(), f"MH class folder not found: {class_folder}"

mhweed_classes = []
for d in sorted(class_folder.iterdir()):
    if d.is_dir():
        n = clean_name(d.name)
        if n:
            mhweed_classes.append(n)

# keep Sicklepod from cotton distinct if duplicated
mhweed_classes = ["Sicklepod Mh" if c == "Sicklepod" else c for c in mhweed_classes]

MASTER = list(dict.fromkeys(cotton_classes + [c for c in mhweed_classes if c not in cotton_classes]))
NC = len(MASTER)

assert NC == 28, f"Expected 28 classes, got {NC}"

COTTON_REMAP = {i: MASTER.index(n) for i, n in enumerate(cotton_classes)}
MHWEED_REMAP = {i: MASTER.index(n) for i, n in enumerate(mhweed_classes)}

CFG["nc"] = NC

log.info(f"Cotton classes ({len(cotton_classes)}): {cotton_classes}")
log.info(f"MH-Weed classes ({len(mhweed_classes)}): {mhweed_classes}")
log.info(f"MASTER ({NC}): {MASTER}")
log.info(f"COTTON_REMAP: {COTTON_REMAP}")
log.info(f"MHWEED_REMAP: {MHWEED_REMAP}")

2026-03-16 18:17:40,278 | INFO | Cotton classes (12): ['Waterhemp', 'MorningGlory', 'Purslane', 'SpottedSpurge', 'Carpetweed', 'Ragweed', 'Eclipta', 'PricklySida', 'PalmerAmaranth', 'Sicklepod', 'Goosegrass', 'Cutleaf']
2026-03-16 18:17:40,279 | INFO | MH-Weed classes (16): ['Kena', 'Lavhala', 'Gajar Gavat', 'Graceful Sandmart', 'Sicklepod Mh', 'Harali', 'Dwarf Cassia', 'Punarnava', 'Lamber Quarter Plant', 'Little Mallow', 'Moti Dudhi', 'Obscure Morning Glory', 'Asian Pigeonwings', 'Bilayat', 'Choti Dudhi', 'Digitaria Sp']
2026-03-16 18:17:40,280 | INFO | MASTER (28): ['Waterhemp', 'MorningGlory', 'Purslane', 'SpottedSpurge', 'Carpetweed', 'Ragweed', 'Eclipta', 'PricklySida', 'PalmerAmaranth', 'Sicklepod', 'Goosegrass', 'Cutleaf', 'Kena', 'Lavhala', 'Gajar Gavat', 'Graceful Sandmart', 'Sicklepod Mh', 'Harali', 'Dwarf Cassia', 'Punarnava', 'Lamber Quarter Plant', 'Little Mallow', 'Moti Dudhi', 'Obscure Morning Glory', 'Asian Pigeonwings', 'Bilayat', 'Choti Dudhi', 'Digitaria Sp']
2026-0

In [4]:
#================================================================
# CELL 4: CLASS UNIFICATION (FIXED)
# IMPORTANT: Clean class names before building master list
#================================================================
# Clean MH-Weed class names (remove trailing _, fix spaces)
def clean_class_name(name):
    # Remove trailing underscore and spaces
    name = name.strip().rstrip('_')
    # Replace underscores with spaces for readability
    name = name.replace('_', ' ')
    # Capitalize first letter of each word
    name = ' '.join(word.capitalize() for word in name.split())
    return name

# Clean both class lists
cotton_classes_clean = cotton_classes  # Cotton names are already clean
mhweed_classes_clean = [clean_class_name(c) for c in mhweed_classes if c.strip()]

log.info(f'Cotton classes (clean): {cotton_classes_clean}')
log.info(f'MH Weed classes (clean): {mhweed_classes_clean}')

# Build master class list — duplicates removed, order preserved
MASTER = list(dict.fromkeys(
    cotton_classes_clean + [c for c in mhweed_classes_clean if c not in cotton_classes_clean]
))
NC = len(MASTER)

# Verify no empty classes
assert all(c.strip() for c in MASTER), "Empty class name found in MASTER list"
assert NC <= 28, f"Expected <=28 classes, got {NC}. Check for duplicates!"

log.info(f'Master class list ({NC} classes): {MASTER}')

# Update config with actual NC
CFG['nc'] = NC

# Remap dicts: original_id -> unified_id
COTTON_REMAP = {i: MASTER.index(c) for i, c in enumerate(cotton_classes_clean)}
MHWEED_REMAP = {i: MASTER.index(c) for i, c in enumerate(mhweed_classes_clean) if c in MASTER}

log.info(f'Cotton remap: {COTTON_REMAP}')
log.info(f'MH Weed remap: {MHWEED_REMAP}')

2026-03-16 18:17:40,301 | INFO | Cotton classes (clean): ['Waterhemp', 'MorningGlory', 'Purslane', 'SpottedSpurge', 'Carpetweed', 'Ragweed', 'Eclipta', 'PricklySida', 'PalmerAmaranth', 'Sicklepod', 'Goosegrass', 'Cutleaf']
2026-03-16 18:17:40,302 | INFO | MH Weed classes (clean): ['Kena', 'Lavhala', 'Gajar Gavat', 'Graceful Sandmart', 'Sicklepod Mh', 'Harali', 'Dwarf Cassia', 'Punarnava', 'Lamber Quarter Plant', 'Little Mallow', 'Moti Dudhi', 'Obscure Morning Glory', 'Asian Pigeonwings', 'Bilayat', 'Choti Dudhi', 'Digitaria Sp']
2026-03-16 18:17:40,304 | INFO | Master class list (28 classes): ['Waterhemp', 'MorningGlory', 'Purslane', 'SpottedSpurge', 'Carpetweed', 'Ragweed', 'Eclipta', 'PricklySida', 'PalmerAmaranth', 'Sicklepod', 'Goosegrass', 'Cutleaf', 'Kena', 'Lavhala', 'Gajar Gavat', 'Graceful Sandmart', 'Sicklepod Mh', 'Harali', 'Dwarf Cassia', 'Punarnava', 'Lamber Quarter Plant', 'Little Mallow', 'Moti Dudhi', 'Obscure Morning Glory', 'Asian Pigeonwings', 'Bilayat', 'Choti Dudhi

In [5]:
#================================================================
# CELL 5: MERGE WITH REAL BOUNDING BOXES — BUGFIXED
#================================================================
import shutil, random
from pathlib import Path
import yaml

print("="*60)
print("MERGING DATASETS WITH REAL BOUNDING BOXES")
print("="*60)

random.seed(SEED)

# Paths
COTTON = Path('/kaggle/input/datasets/jawadulkarim117/cotton-weed-12-class/cotton_weed')
DATA_ROOT = Path('/kaggle/input/datasets/sayalis069/mh-weed16/MH-Weed16An Indian Multiclass Annotated Weed Dataset for Computer Vision Tasks/MH-Weed16')
YOLO_LABELS = DATA_ROOT / 'Crop with Weeds' / 'intel Real Sense Depth_Annotations' / 'intel Real Sense Depth_Annotations' / 'YOLO_darknet'

MHWEED_IMG_DIRS = [
    DATA_ROOT / 'Crop with Weeds' / 'Canon Camera_Clicks' / 'Canon Camera_Clicks',
    DATA_ROOT / 'Crop with Weeds' / 'iPhone_Clicks' / 'iPhone_Clicks',
    DATA_ROOT / 'Crop with Weeds' / 'intel Real Sense Depth_Clicks' / 'intel Real Sense Depth_Clicks',
]

# ---- CRITICAL FIX 1: reset old merged data ----
if MERGED.exists():
    shutil.rmtree(MERGED)

for split in ['train', 'val', 'test']:
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

print(f"COTTON path: {COTTON} (exists: {COTTON.exists()})")
print(f"YOLO labels: {len(list(YOLO_LABELS.glob('*.txt')))} files")

def valid_img_list(img_dir: Path):
    exts = ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']
    out = []
    for e in exts:
        out.extend(img_dir.glob(e))
    return out

def remap_label_lines(lines, remap_dict):
    mapped = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        old_cls = int(float(parts[0]))
        if old_cls not in remap_dict:
            continue
        x, y, w, h = map(float, parts[1:])
        if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            continue
        new_cls = remap_dict[old_cls]
        mapped.append(f"{new_cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    return mapped

# 1) MERGE COTTON (strict image-label pairing)
def merge_cotton():
    count = 0
    for split_name, folder_name in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
        img_dir = COTTON / folder_name / 'images'
        lbl_dir = COTTON / folder_name / 'labels'
        if not img_dir.exists() or not lbl_dir.exists():
            print(f"⚠ Missing cotton split dirs for {split_name}")
            continue

        imgs = valid_img_list(img_dir)
        print(f"  Cotton {split_name}: {len(imgs)} raw images")

        kept = 0
        for img_path in imgs:
            lbl = lbl_dir / f'{img_path.stem}.txt'
            if not lbl.exists():
                continue

            lines = lbl.read_text().strip().splitlines()
            new_lines = remap_label_lines(lines, COTTON_REMAP)
            if len(new_lines) == 0:
                continue

            stem = f'c_{count:08d}'
            out_img = MERGED / split_name / 'images' / f'{stem}{img_path.suffix.lower()}'
            out_lbl = MERGED / split_name / 'labels' / f'{stem}.txt'

            shutil.copy2(img_path, out_img)
            out_lbl.write_text('\n'.join(new_lines))
            count += 1
            kept += 1

        print(f"    ✅ kept labeled: {kept}")
    return count

# 2) MERGE MH-WEED (only YOLO-labeled pairs)
def merge_mhweed_real():
    all_pairs = []
    for img_dir in MHWEED_IMG_DIRS:
        if not img_dir.exists():
            print(f"⚠ Not found: {img_dir}")
            continue

        images = valid_img_list(img_dir)
        matched = 0
        for img_path in images:
            lbl_path = YOLO_LABELS / f'{img_path.stem}.txt'
            if lbl_path.exists():
                all_pairs.append((img_path, lbl_path))
                matched += 1

        print(f"✅ {img_dir.name}: images={len(images)}, matched={matched}")

    if len(all_pairs) == 0:
        raise ValueError("No MH-Weed image-label pairs found.")

    random.shuffle(all_pairs)
    n = len(all_pairs)
    splits = {
        'train': all_pairs[:int(0.8*n)],
        'val':   all_pairs[int(0.8*n):int(0.9*n)],
        'test':  all_pairs[int(0.9*n):]
    }

    count = 0
    for split_name, pairs in splits.items():
        kept = 0
        for img_path, lbl_path in pairs:
            lines = lbl_path.read_text().strip().splitlines()
            new_lines = remap_label_lines(lines, MHWEED_REMAP)
            if len(new_lines) == 0:
                continue

            stem = f'm_{count:08d}'
            out_img = MERGED / split_name / 'images' / f'{stem}{img_path.suffix.lower()}'
            out_lbl = MERGED / split_name / 'labels' / f'{stem}.txt'

            shutil.copy2(img_path, out_img)
            out_lbl.write_text('\n'.join(new_lines))
            count += 1
            kept += 1

        print(f"  MH {split_name}: ✅ kept labeled {kept}")
    return count

# RUN
print("\n🔄 Merging Cotton...")
c_count = merge_cotton()
print(f"✅ Cotton kept: {c_count}")

print("\n🔄 Merging MH-Weed...")
m_count = merge_mhweed_real()
print(f"✅ MH-Weed kept: {m_count}")

total = c_count + m_count
print(f"\n{'='*60}")
print(f"📊 TOTAL kept labeled images: {total}")

# ---- CRITICAL FIX 2: enforce images == labels ----
for split in ['train', 'val', 'test']:
    n_img = len(list((MERGED / split / 'images').glob('*')))
    n_lbl = len(list((MERGED / split / 'labels').glob('*.txt')))
    print(f"  {split}: {n_img} images, {n_lbl} labels")
    assert n_img == n_lbl, f"{split} mismatch: {n_img} images vs {n_lbl} labels"

yaml.dump({
    'path': str(MERGED),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    NC,
    'names': {i: MASTER[i] for i in range(NC)}
}, open(MERGED / 'data.yaml', 'w'), default_flow_style=False)

print(f"\n✅ Ready: {NC} classes, {total} labeled images")

MERGING DATASETS WITH REAL BOUNDING BOXES
COTTON path: /kaggle/input/datasets/jawadulkarim117/cotton-weed-12-class/cotton_weed (exists: True)
YOLO labels: 6656 files

🔄 Merging Cotton...
  Cotton train: 12250 raw images
    ✅ kept labeled: 12249
  Cotton val: 1342 raw images
    ✅ kept labeled: 1342
  Cotton test: 2391 raw images
    ✅ kept labeled: 2391
✅ Cotton kept: 15982

🔄 Merging MH-Weed...
✅ Canon Camera_Clicks: images=345, matched=0
✅ iPhone_Clicks: images=576, matched=0
✅ intel Real Sense Depth_Clicks: images=6656, matched=6656
  MH train: ✅ kept labeled 5317
  MH val: ✅ kept labeled 665
  MH test: ✅ kept labeled 666
✅ MH-Weed kept: 6648

📊 TOTAL kept labeled images: 22630
  train: 17566 images, 17566 labels
  val: 2007 images, 2007 labels
  test: 3057 images, 3057 labels

✅ Ready: 28 classes, 22630 labeled images


In [6]:
# ================================================================
# CELL 5.5: MERGE RARE CLASSES (CRITICAL FOR 70+ mAP)
# ================================================================

from pathlib import Path
import yaml
from collections import Counter

print("="*70)
print("🔧 MERGING RARE CLASSES FOR BALANCE")
print("="*70)

# Load current data.yaml
with open(YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Count current distribution
print("Analyzing class distribution...")
class_counts = Counter()
for split in ['train', 'val']:
    label_dir = MERGED / split / 'labels'
    for txt_file in label_dir.glob('*.txt'):
        with open(txt_file) as f:
            for line in f:
                cls = int(line.split()[0])
                class_counts[cls] += 1

print("\nCurrent class distribution:")
for cls_id, count in sorted(class_counts.items()):
    status = "✅ Keep" if count >= 400 else "🔴 MERGE"
    print(f"  Class {cls_id}: {count:5d} images [{status}]")

# Define rare classes (< 400 images) - ADJUST THESE BASED ON YOUR COUNTS
# Based on your previous output, these were the rare ones:
RARE_CLASSES = [18, 19, 23, 25, 26, 27]  # Adjust if your indices differ
MERGE_TARGET = 18  # All rare classes become class 18 (OtherWeed)

print(f"\nMerging classes {RARE_CLASSES} into Class {MERGE_TARGET} (OtherWeed)")
print(f"Before: 28 classes | After: {28 - len(RARE_CLASSES) + 1} classes")

# Create remapping dictionary
OLD_TO_NEW = {}
for i in range(28):
    if i in RARE_CLASSES:
        OLD_TO_NEW[i] = MERGE_TARGET
    else:
        # Shift classes after merge target to fill gaps
        shift = sum(1 for r in RARE_CLASSES if r < i and r != MERGE_TARGET)
        OLD_TO_NEW[i] = i - shift

print(f"\nRemapping: {OLD_TO_NEW}")

# Function to remap label files
def remap_labels(split):
    label_dir = MERGED / split / 'labels'
    remapped_count = 0
    
    for txt_file in label_dir.glob('*.txt'):
        lines = []
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    old_cls = int(parts[0])
                    new_cls = OLD_TO_NEW[old_cls]
                    parts[0] = str(new_cls)
                    lines.append(' '.join(parts))
        
        # Rewrite file
        with open(txt_file, 'w') as f:
            f.write('\n'.join(lines) + '\n' if lines else '')
        remapped_count += 1
    
    return remapped_count

# Apply remapping
print("\nRemapping labels...")
for split in ['train', 'val', 'test']:
    count = remap_labels(split)
    print(f"  {split}: {count} files remapped")

# Update class names (remove merged classes, add OtherWeed)
old_names = data_cfg.get('names', {})
new_names = {}
new_idx = 0

for old_idx, name in old_names.items():
    old_idx = int(old_idx)
    if old_idx in RARE_CLASSES and old_idx != MERGE_TARGET:
        continue  # Skip rare classes (except the merge target)
    elif old_idx == MERGE_TARGET:
        new_names[new_idx] = "OtherWeed"  # Rename merge target
        new_idx += 1
    else:
        new_names[new_idx] = name
        new_idx += 1

# Update data.yaml
data_cfg['names'] = new_names
data_cfg['nc'] = len(new_names)

with open(YAML_PATH, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\n✅ Updated {YAML_PATH}")
print(f"New class count: {len(new_names)}")
print(f"Classes: {list(new_names.values())}")

# Verify new distribution
print("\nVerifying new distribution...")
new_counts = Counter()
for split in ['train']:
    label_dir = MERGED / split / 'labels'
    for txt_file in label_dir.glob('*.txt'):
        with open(txt_file) as f:
            for line in f:
                cls = int(line.split()[0])
                new_counts[cls] += 1

print("New distribution:")
for cls_id, count in sorted(new_counts.items()):
    print(f"  Class {cls_id}: {count} images")

min_count = min(new_counts.values())
print(f"\nMinimum class count: {min_count}")
if min_count >= 400:
    print("✅ All classes have 400+ images! Ready for 70+ mAP training.")
else:
    print("⚠️ Still have small classes. Consider merging more.")

🔧 MERGING RARE CLASSES FOR BALANCE
Analyzing class distribution...

Current class distribution:
  Class 0:  4538 images [✅ Keep]
  Class 1:  2958 images [✅ Keep]
  Class 2:  2080 images [✅ Keep]
  Class 3:  1725 images [✅ Keep]
  Class 4:  1649 images [✅ Keep]
  Class 5:  1871 images [✅ Keep]
  Class 6:  1948 images [✅ Keep]
  Class 7:  1118 images [✅ Keep]
  Class 8:   825 images [✅ Keep]
  Class 9:   585 images [✅ Keep]
  Class 10:   481 images [✅ Keep]
  Class 11:   275 images [🔴 MERGE]
  Class 12:  5833 images [✅ Keep]
  Class 13: 20410 images [✅ Keep]
  Class 14: 20122 images [✅ Keep]
  Class 15:  1954 images [✅ Keep]
  Class 16:  1612 images [✅ Keep]
  Class 17:  2090 images [✅ Keep]
  Class 18:   166 images [🔴 MERGE]
  Class 19:   185 images [🔴 MERGE]
  Class 20:   757 images [✅ Keep]
  Class 21:  1288 images [✅ Keep]
  Class 22:   408 images [✅ Keep]
  Class 23:   107 images [🔴 MERGE]
  Class 24:   351 images [🔴 MERGE]
  Class 25:   157 images [🔴 MERGE]
  Class 26:   242 images

In [7]:
# ================================================================
# CELL 5.6: FINAL MERGE (Classes 11, 21, 22 → OtherWeed)
# ================================================================

from pathlib import Path
import yaml
from collections import Counter

print("="*70)
print("🔧 FINAL MERGE: Getting all classes to 750+ images")
print("="*70)

# Merge remaining small classes into Class 18 (OtherWeed)
ADDITIONAL_RARE = [11, 21, 22]  # Cutleaf, Moti Dudhi, Asian Pigeonwings
MERGE_TARGET = 18

print(f"Merging classes {ADDITIONAL_RARE} into Class {MERGE_TARGET}")

# Remap function
def final_remap(split):
    label_dir = MERGED / split / 'labels'
    for txt_file in label_dir.glob('*.txt'):
        lines = []
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls = int(parts[0])
                    if cls in ADDITIONAL_RARE:
                        cls = MERGE_TARGET
                    # Adjust indices for removed classes (21,22 shift down)
                    if cls > 11 and cls < 18:
                        cls -= 1  # Shift for class 11 removal
                    elif cls > 18:  # 19,20 become 18,19
                        cls -= 3  # Shift for 11,21,22 removal
                    parts[0] = str(cls)
                    lines.append(' '.join(parts))
        
        with open(txt_file, 'w') as f:
            f.write('\n'.join(lines) + '\n' if lines else '')

for split in ['train', 'val', 'test']:
    final_remap(split)
    print(f"  {split} remapped")

# Update YAML
with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

names = cfg['names']
new_names = {}
idx_map = {}

# Build new name list (remove 11, 21, 22)
new_idx = 0
for old_idx in range(23):
    if old_idx in ADDITIONAL_RARE:
        continue
    if old_idx == MERGE_TARGET:
        new_names[new_idx] = "OtherWeed"  # Keep as OtherWeed
    else:
        new_names[new_idx] = names[old_idx]
    idx_map[old_idx] = new_idx
    new_idx += 1

cfg['names'] = new_names
cfg['nc'] = len(new_names)

with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\n✅ Now 20 classes total")
print(f"Classes: {list(new_names.values())}")

# Verify
print("\nFinal distribution:")
final_counts = Counter()
for split in ['train']:
    for txt_file in (MERGED / split / 'labels').glob('*.txt'):
        with open(txt_file) as f:
            for line in f:
                final_counts[int(line.split()[0])] += 1

for cls_id, count in sorted(final_counts.items()):
    print(f"  Class {cls_id}: {count} images")

print(f"\nMinimum count: {min(final_counts.values())} ✅")
print("Ready for 70+ mAP training!")

🔧 FINAL MERGE: Getting all classes to 750+ images
Merging classes [11, 21, 22] into Class 18
  train remapped
  val remapped
  test remapped

✅ Now 20 classes total
Classes: ['Waterhemp', 'MorningGlory', 'Purslane', 'SpottedSpurge', 'Carpetweed', 'Ragweed', 'Eclipta', 'PricklySida', 'PalmerAmaranth', 'Sicklepod', 'Goosegrass', 'Kena', 'Lavhala', 'Gajar Gavat', 'Graceful Sandmart', 'Sicklepod Mh', 'Harali', 'OtherWeed', 'Lamber Quarter Plant', 'Little Mallow']

Final distribution:
  Class 0: 4037 images
  Class 1: 2678 images
  Class 2: 1816 images
  Class 3: 1632 images
  Class 4: 1491 images
  Class 5: 1692 images
  Class 6: 1704 images
  Class 7: 1023 images
  Class 8: 762 images
  Class 9: 527 images
  Class 10: 430 images
  Class 11: 5134 images
  Class 12: 18249 images
  Class 13: 17829 images
  Class 14: 1727 images
  Class 15: 1426 images
  Class 16: 2514 images
  Class 17: 1134 images
  Class 18: 1686 images

Minimum count: 430 ✅
Ready for 70+ mAP training!


In [8]:
#================================================================
# CELL 6: LOAD TRAINED TEACHER WEIGHTS
#================================================================
from ultralytics import YOLO

TEACHER_WEIGHTS = Path(CFG['teacher_weights'])
assert TEACHER_WEIGHTS.exists(), f'Teacher weights not found: {TEACHER_WEIGHTS}'

teacher_yolo = YOLO(str(TEACHER_WEIGHTS))
log.info(f'✅ Teacher loaded from: {TEACHER_WEIGHTS}')
print(f'✅ Teacher loaded: {TEACHER_WEIGHTS}')
print(f'   mAP50=0.792 | mAP50-95=0.603 | 28 classes')

2026-03-16 18:25:52,828 | INFO | ✅ Teacher loaded from: /kaggle/input/datasets/rahu12345/teacher/best.pt


✅ Teacher loaded: /kaggle/input/datasets/rahu12345/teacher/best.pt
   mAP50=0.792 | mAP50-95=0.603 | 28 classes


In [9]:
#================================================================
# CELL 4: LOAD FROZEN TEACHER (YOLOv8l) - FIXED
#================================================================
import torch
import torch.nn as nn
from ultralytics import YOLO

class FrozenTeacher(nn.Module):
    """
    Loads trained YOLO teacher and freezes it.
    Hooks at layers 4, 6, 9 capture intermediate features for KD.
    """
    def __init__(self, weights_path):
        super().__init__()
        self._feats = {}

        # IMPORTANT: do not keep YOLO object as child module
        y = YOLO(str(weights_path))
        self.model = y.model
        del y

        # Freeze teacher params
        for p in self.model.parameters():
            p.requires_grad = False

        # Keep underlying model in eval mode
        self.model.eval()

        # Register hooks
        self._register_hooks()

        total = sum(p.numel() for p in self.model.parameters())
        print(f"✅ Teacher loaded: {total/1e6:.1f}M params (frozen)")

    def _register_hooks(self):
        def hook_fn(name):
            def fn(module, inp, out):
                feat = out[0] if isinstance(out, tuple) else out
                self._feats[name] = feat.detach()
            return fn

        for idx in [4, 6, 9]:
            self.model.model[idx].register_forward_hook(hook_fn(f"l{idx}"))

    def forward(self, x):
        self._feats.clear()
        with torch.no_grad():
            return self.model(x)

    def features(self):
        return dict(self._feats)

# Load teacher
teacher = FrozenTeacher(CFG['teacher_weights']).to(device)

# Test
with torch.no_grad():
    test = torch.randn(1, 3, 640, 640, device=device)
    _ = teacher(test)
feats = teacher.features()

print(f"✅ Feature layers: {list(feats.keys())}")
for k, v in feats.items():
    print(f"{k}: {tuple(v.shape)}")

✅ Teacher loaded: 43.7M params (frozen)
✅ Feature layers: ['l4', 'l6', 'l9']
l4: (1, 256, 80, 80)
l6: (1, 512, 40, 40)
l9: (1, 512, 20, 20)


In [10]:
# ================================================================
# CELL 8 (FIXED): USE YOLOv8s TO MATCH CHECKPOINT
# ================================================================
import torch
import torch.nn as nn
from ultralytics.nn.tasks import DetectionModel, yaml_model_load

HOOK_LAYERS = [4, 6, 9]

class FrozenTeacher(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self._feats = {}

        from ultralytics import YOLO
        y = YOLO(str(weights_path))
        self.model = y.model
        del y

        for p in self.model.parameters():
            p.requires_grad = False
        self.model.eval()
        self._register_hooks()

    def _register_hooks(self):
        def hook_fn(name):
            def fn(module, inp, out):
                self._feats[name] = (out[0] if isinstance(out, tuple) else out).detach()
            return fn
        for i in HOOK_LAYERS:
            self.model.model[i].register_forward_hook(hook_fn(f"l{i}"))

    def forward(self, x):
        self._feats.clear()
        with torch.no_grad():
            return self.model(x)

    def features(self):
        return dict(self._feats)


class Student(nn.Module):
    def __init__(self, nc=28):
        """
        Build YOLOv8s from YAML config to match checkpoint
        """
        super().__init__()
        self._feats = {}

        # ✅ Use yolov8s.yaml (matches your checkpoint)
        self.model = DetectionModel(cfg="yolov8s.yaml", ch=3, nc=nc)
        
        self._adapt_detect_head(nc)
        self.model.nc = nc
        self.model.names = {i: str(i) for i in range(nc)}
        self._register_hooks()

    def _adapt_detect_head(self, nc):
        det = self.model.model[-1]
        det.nc = nc

        for b in det.cv3:
            old = b[-1]
            new = nn.Conv2d(
                old.in_channels, nc,
                kernel_size=old.kernel_size,
                stride=old.stride,
                padding=old.padding,
                bias=True
            ).to(old.weight.device)
            nn.init.normal_(new.weight, mean=0.0, std=0.01)
            nn.init.constant_(new.bias, -4.5)
            b[-1] = new

        det.no = det.nc + det.reg_max * 4

    def _register_hooks(self):
        def hook_fn(name):
            def fn(module, inp, out):
                self._feats[name] = out[0] if isinstance(out, tuple) else out
            return fn
        for i in HOOK_LAYERS:
            self.model.model[i].register_forward_hook(hook_fn(f"l{i}"))

    def forward(self, x):
        self._feats.clear()
        return self.model(x)

    def features(self):
        return dict(self._feats)


class FeatureAdapters(nn.Module):
    def __contains__(self, key):
        return key in self.adapters
    
    def __init__(self, s_ch, t_ch):
        super().__init__()
        self.adapters = nn.ModuleDict({
            k: nn.Conv2d(s_ch[k], t_ch[k], 1, bias=True) for k in s_ch.keys()
        })
        for m in self.adapters.values():
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def __getitem__(self, key):
        return self.adapters[key]

    def keys(self):
        return self.adapters.keys()


# ✅ INSTANTIATE with YOLOv8s
teacher = FrozenTeacher(CFG['teacher_weights']).to(device)
student = Student(nc=NC).to(device)

print("✅ Student built from YOLOv8s (matches checkpoint)")

# Test shapes
with torch.no_grad():
    x = torch.randn(2, 3, CFG['img_size'], CFG['img_size'], device=device)
    teacher.model.eval()
    student.model.eval()
    _ = teacher(x)
    t_feats = teacher.features()
    _ = student(x)
    s_feats = student.features()

s_ch = {k: v.shape[1] for k, v in s_feats.items()}
t_ch = {k: v.shape[1] for k, v in t_feats.items()}
adapters = FeatureAdapters(s_ch, t_ch).to(device)

print("Student channels:", s_ch)
print("Teacher channels:", t_ch)
print("✅ Teacher/Student/Adapters ready")

Overriding model.yaml nc=80 with nc=28

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TrueKDLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.05, gamma=0.0, temperature=4.0, nc=27):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.T = temperature
        self.nc = nc
        self.det_criterion = None

    def set_criterion(self, model):
        from types import SimpleNamespace
        from ultralytics.utils.loss import v8DetectionLoss
        
        model.args = SimpleNamespace(
            box=7.5, cls=0.5, dfl=1.5,
            pose=12.0, kobj=1.0,
            label_smoothing=0.0, nc=self.nc,
            overlap_mask=True, mask_ratio=4
        )
        self.det_criterion = v8DetectionLoss(model)

    def detection_loss(self, s_out, targets):
        if self.det_criterion is None:
            raise RuntimeError("Call set_criterion() first")
        
        if targets.numel() == 0:
            return torch.tensor(0.0, device=next(self.det_criterion.model.parameters()).device)
        
        batch = {
            "batch_idx": targets[:, 0].long(),
            "cls": targets[:, 1].long(),
            "bboxes": targets[:, 2:6],
        }
        
        loss, _ = self.det_criterion(s_out, batch)
        return loss.mean() if torch.is_tensor(loss) and loss.ndim > 0 else loss

    def feature_loss(self, s_feats, t_feats, adapters):
        total, count = 0.0, 0
        for k in ["l4", "l6", "l9"]:
            if k in s_feats and k in t_feats and k in adapters:
                total = total + F.mse_loss(
                    adapters[k](s_feats[k]), 
                    t_feats[k]
                )
                count += 1
        
        if count == 0:
            return torch.tensor(0.0, device=next(adapters.parameters()).device)
        
        return total / count

    def kl_loss(self, s_out, t_out, targets):
        # 🐛 FIX: Use targets.device instead of trying to parse s_out!
        # targets is guaranteed to be a PyTorch tensor
        return torch.tensor(0.0, device=targets.device)

    def forward(self, s_out, t_out, s_feats, t_feats, adapters, targets):
        d = self.detection_loss(s_out, targets)
        f = self.feature_loss(s_feats, t_feats, adapters)
        k = self.kl_loss(s_out, t_out, targets) # 🐛 FIX: Pass targets to kl_loss
        
        total = self.alpha * d + self.beta * f + self.gamma * k
        
        return total, float(d.detach()), float(f.detach()), float(k.detach())

kd_loss = TrueKDLoss(alpha=1.0, beta=0.05, gamma=0.0, temperature=4.0, nc=NC).to(device)
kd_loss.set_criterion(student.model)
print("\nKD loss initialized")



KD loss initialized


In [12]:
# ================================================================
# CELL 7 (FIXED): AUGMENTATIONS FOR YOLO (NO IMAGENET NORMALIZE)
# ================================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_transforms(train=True):
    if train:
        return A.Compose([
            A.Resize(CFG['img_size'], CFG['img_size']),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
            A.GaussNoise(std_range=(0.02, 0.08), p=0.15),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), rotate=(-15, 15), p=0.25),

            # IMPORTANT: scale only to [0,1], no mean/std shift
            A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['cls'], min_visibility=0.2))
    else:
        return A.Compose([
            A.Resize(CFG['img_size'], CFG['img_size']),
            A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['cls']))

train_tf = get_transforms(True)
val_tf   = get_transforms(False)
print("✅ Transforms ready (YOLO scale)")

✅ Transforms ready (YOLO scale)


In [13]:
# ================================================================
# CELL X (FIXED): AUGMENTATIONS + DATASET + DATALOADER FOR YOLO KD
# (NO IMAGENET NORMALIZATION; OUTPUT IMAGE RANGE ~ [0,1])
# ================================================================
from pathlib import Path
from collections import Counter
import cv2
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2

# -----------------------------
# 1) Transforms (FIXED)
# -----------------------------
IMG = CFG['img_size']

train_tf = A.Compose(
    [
        A.Resize(IMG, IMG),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.HueSaturationValue(p=0.2),
        # IMPORTANT: no ImageNet mean/std normalize
        # Keep only pixel scaling to [0,1]:
        A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['cls'],
        min_visibility=0.0,
        check_each_transform=False
    )
)

val_tf = A.Compose(
    [
        A.Resize(IMG, IMG),
        A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['cls'],
        min_visibility=0.0,
        check_each_transform=False
    )
)

# -----------------------------
# 2) Dataset
# -----------------------------
class WeedDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, transform):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.transform = transform

        exts = ("*.jpg","*.jpeg","*.JPG","*.JPEG","*.png","*.PNG")
        imgs = []
        for e in exts:
            imgs += list(self.img_dir.glob(e))
        imgs = sorted(imgs)

        # strict image-label pairing
        self.samples = []
        for img in imgs:
            lbl = self.lbl_dir / f"{img.stem}.txt"
            if lbl.exists():
                self.samples.append((img, lbl))

        self.primary_cls = self._primary_classes()

    def _primary_classes(self):
        out = []
        for _, lbl in self.samples:
            txt = lbl.read_text().strip()
            if txt:
                out.append(int(float(txt.splitlines()[0].split()[0])))
        return out if out else [0]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]

        img = cv2.imread(str(img_path))
        if img is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes, cls = [], []
        txt = lbl_path.read_text().strip()
        if txt:
            for line in txt.splitlines():
                p = line.split()
                if len(p) != 5:
                    continue

                c = int(float(p[0]))
                x, y, w, h = map(float, p[1:])

                eps = 1e-4
                x = min(max(x, eps), 1 - eps)
                y = min(max(y, eps), 1 - eps)
                w = min(max(w, eps), 1 - eps)
                h = min(max(h, eps), 1 - eps)
                if w <= eps or h <= eps:
                    continue

                # keep center valid after clamps
                x = min(max(x, w / 2 + eps), 1 - w / 2 - eps)
                y = min(max(y, h / 2 + eps), 1 - h / 2 - eps)

                boxes.append([x, y, w, h])
                cls.append(c)

        transformed = self.transform(image=img, bboxes=boxes, cls=cls)
        img_t = transformed["image"]  # tensor float32, CHW, ~[0,1]
        bbs = transformed["bboxes"]
        ccs = transformed["cls"]

        if len(bbs) == 0:
            return img_t, torch.zeros((0, 6), dtype=torch.float32)

        targets = torch.zeros((len(bbs), 6), dtype=torch.float32)
        for i, (b, c) in enumerate(zip(bbs, ccs)):
            targets[i] = torch.tensor([0, float(c), float(b[0]), float(b[1]), float(b[2]), float(b[3])], dtype=torch.float32)

        return img_t, targets

# -----------------------------
# 3) Collate
# -----------------------------
def collate_fn(batch):
    imgs, targets = zip(*batch)
    imgs = torch.stack(imgs, dim=0)

    merged = []
    for i, t in enumerate(targets):
        if t.numel() == 0:
            continue
        tt = t.clone()
        tt[:, 0] = i
        merged.append(tt)

    if len(merged) == 0:
        return imgs, torch.zeros((0, 6), dtype=torch.float32)

    return imgs, torch.cat(merged, dim=0)

# -----------------------------
# 4) Build datasets/loaders
# -----------------------------
train_ds = WeedDataset(MERGED/'train'/'images', MERGED/'train'/'labels', train_tf)
val_ds   = WeedDataset(MERGED/'val'/'images',   MERGED/'val'/'labels',   val_tf)

def get_sampler(dataset):
    cnt = Counter(dataset.primary_cls)
    weights = [1.0 / max(cnt[c], 1) for c in dataset.primary_cls]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    sampler=get_sampler(train_ds),
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True
)

print(f"✅ Train samples: {len(train_ds)}")
print(f"✅ Val samples:   {len(val_ds)}")

# -----------------------------
# 5) Range sanity check
# -----------------------------
imgs, t = next(iter(train_loader))
print("image range:", float(imgs.min()), float(imgs.max()), "mean:", float(imgs.mean()), "std:", float(imgs.std()))
print("targets shape:", tuple(t.shape))
print("✅ Expect min~0 and max~1")

✅ Train samples: 17566
✅ Val samples:   2007
image range: 0.0 1.0 mean: 0.43615153431892395 std: 0.23878715932369232
targets shape: (53, 6)
✅ Expect min~0 and max~1


In [14]:
#================================================================
# CELL 9: OPTIMIZER + SCHEDULER
#================================================================
# Optimizer: different LR for student vs adapters
optimizer = torch.optim.AdamW([
    {'params': student.parameters(), 'lr': CFG['kd_lr'], 'weight_decay': CFG['weight_decay']},
    {'params': adapters.parameters(), 'lr': CFG['kd_lr'] * 2, 'weight_decay': 0}  # Faster learning for new layers
])

# Cosine annealing LR scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['kd_epochs'])

# Mixed precision for faster training
scaler = torch.amp.GradScaler('cuda')

print(f"✅ Optimizer: AdamW")
print(f"✅ Student LR: {CFG['kd_lr']}")
print(f"✅ Adapter LR: {CFG['kd_lr'] * 2}")
print(f"✅ Scheduler: CosineAnnealing")
print(f"✅ Mixed precision: Enabled")

✅ Optimizer: AdamW
✅ Student LR: 0.001
✅ Adapter LR: 0.002
✅ Scheduler: CosineAnnealing
✅ Mixed precision: Enabled


In [15]:
# ================================================================
# CELL 13 (FIXED): VALIDATION FUNCTION
# ================================================================

@torch.no_grad()
def validate():
    """Validate student model on validation set"""
    import torch
    from pathlib import Path
    
    try:
        # 1. Save current student state
        tmp_ckpt = OUTPUT / "tmp_student_val.pt"
        torch.save(student.model.state_dict(), tmp_ckpt)
        print("   ⏳ Validating...")
        
        # 2. Build a fresh DetectionModel for validation
        val_model = Student(nc=NC).to(device)
        
        # 3. Load the saved weights
        val_model.model.load_state_dict(torch.load(tmp_ckpt, map_location='cpu'), strict=False)
        val_model.model.eval()
        
        # 4. Setup model for YOLO inference
        m = val_model.model
        m.nc = NC
        m.names = {i: MASTER[i] for i in range(NC)}
        
        # 5. Wrap in YOLO object
        from ultralytics import YOLO
        y = YOLO('yolov8s.pt')
        y.model = m
        
        # 6. Run validation
        results = y.val(
            data=str(YAML_PATH),
            imgsz=CFG['img_size'],
            batch=CFG['batch_size'],
            device=0 if "cuda" in str(device) else "cpu",
            verbose=False,
            plots=False
        )
        
        # 7. Extract metrics
        map50 = float(results.box.map50) if hasattr(results, 'box') else 0.0
        map5095 = float(results.box.map) if hasattr(results, 'box') else 0.0
        
        # Cleanup
        if tmp_ckpt.exists():
            tmp_ckpt.unlink()
        
        return map50, map5095
        
    except Exception as e:
        print(f"   ⚠️ Validation error: {type(e).__name__}: {str(e)[:50]}")
        return 0.0, 0.0

# Test the validation function
print("Testing validation function...")
map50, map5095 = validate()
print(f"✅ Validation test: mAP50={map50:.4f}, mAP50-95={map5095:.4f}")

Testing validation function...
   ⏳ Validating...
Overriding model.yaml nc=80 with nc=28

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]          

In [16]:
# ================================================================
# KD PREFLIGHT SANITY CHECK (FIXED)
# ================================================================
import torch


def kd_preflight_check(student, teacher, adapters, kd_loss, train_loader, device):

    # eval for shape/loss sanity
    student.model.eval()
    teacher.model.eval()
    adapters.eval()

    imgs, targets = next(iter(train_loader))
    imgs = imgs.to(device).float()
    targets = targets.to(device)

    # forward (sanity pass)
    with torch.no_grad():
        t_out = teacher(imgs)
        t_feats = teacher.features()

        s_out = student(imgs)
        s_feats = student.features()

    # teacher wrapper already no_grad
    # eval-mode forward for sanity only
    print("=== KD PREFLIGHT ===")
    print(f"imgs: {tuple(imgs.shape)} dtype={imgs.dtype} min={imgs.min().item()}")
    print(f"targets: {tuple(targets.shape)}")

    print("student feat keys:", sorted(list(s_feats.keys())))
    print("teacher feat keys:", sorted(list(t_feats.keys())))
    print("adapter keys:", sorted(list(adapters.keys())))

    required = ["l4", "l6", "l9"]

    for k in required:
        ok = (k in s_feats) and (k in t_feats) and (k in adapters)
        print(f"[{k}] present in s/t/adapters -> {ok}")

        if ok:
            s = s_feats[k]
            t = t_feats[k]
            a = adapters[k](s)

            print(f" s:{tuple(s.shape)} t:{tuple(t.shape)} a(s):{tuple(a.shape)}")

            if a.shape != t.shape:
                print(f"\nSHAPE MISMATCH at {k}: adapter(s) != teacher")
                return False

    # ---------------- detection-only loss ----------------
    kd_loss.alpha, kd_loss.beta, kd_loss.gamma = 1.0, 0.0, 0.0
    total, d, f, k = kd_loss(s_out, t_out, s_feats, t_feats, adapters, targets)

    print(f"det-only: total={float(total):.4f}, det={d:.4f}, feat={f:.4f}, kl={k:.4f}")

    if not torch.isfinite(total):
        print("\nNon-finite det-only loss")
        return False

    # ---------------- feature-KD loss ----------------
    kd_loss.alpha, kd_loss.beta, kd_loss.gamma = 1.0, 0.05, 0.0
    total2, d2, f2, k2 = kd_loss(s_out, t_out, s_feats, t_feats, adapters, targets)

    print(f"feat-kd : total={float(total2):.4f}, det={d2:.4f}, feat={f2:.4f}, kl={k2:.4f}")

    if not torch.isfinite(total2):
        print("\nNon-finite feature-KD loss")
        return False

    if f2 < 0:
        print("\nFeature loss should not be negative")
        return False

    # ---------------- gradient flow test ----------------
    student.model.train()
    adapters.train()

    for p in student.model.parameters():
        if p.grad is not None:
            p.grad = None

    for p in adapters.parameters():
        if p.grad is not None:
            p.grad = None

    # recompute with grad enabled for student/adapters
    with torch.no_grad():
        t_out = teacher(imgs)
        t_feats = teacher.features()

    s_out = student(imgs)
    s_feats = student.features()

    kd_loss.alpha, kd_loss.beta, kd_loss.gamma = 1.0, 0.05, 0.0
    total3, d3, f3, k3 = kd_loss(s_out, t_out, s_feats, t_feats, adapters, targets)

    total3.backward()

    grad_student = 0.0
    grad_adapters = 0.0

    for p in student.model.parameters():
        if p.grad is not None:
            grad_student += float(p.grad.abs().mean())

    for p in adapters.parameters():
        if p.grad is not None:
            grad_adapters += float(p.grad.abs().mean())

    print(f"grad mean sum -> student={grad_student:.6f}, adapters={grad_adapters:.6f}")

    if grad_adapters == 0.0:
        print("\nNo gradients in adapters")
        return False

    print("\nKD preflight PASSED")
    return True


ok = kd_preflight_check(student, teacher, adapters, kd_loss, train_loader, device)
print("READY_TO_TRAIN =", ok)

=== KD PREFLIGHT ===
imgs: (16, 3, 640, 640) dtype=torch.float32 min=0.0
targets: (98, 6)
student feat keys: ['l4', 'l6', 'l9']
teacher feat keys: ['l4', 'l6', 'l9']
adapter keys: ['l4', 'l6', 'l9']
[l4] present in s/t/adapters -> True
 s:(16, 128, 80, 80) t:(16, 256, 80, 80) a(s):(16, 256, 80, 80)
[l6] present in s/t/adapters -> True
 s:(16, 256, 40, 40) t:(16, 512, 40, 40) a(s):(16, 512, 40, 40)
[l9] present in s/t/adapters -> True
 s:(16, 512, 20, 20) t:(16, 512, 20, 20) a(s):(16, 512, 20, 20)


/tmp/ipykernel_23/2197410291.py:57: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"det-only: total={float(total):.4f}, det={d:.4f}, feat={f:.4f}, kl={k:.4f}")


det-only: total=400.9318, det=400.9318, feat=0.1496, kl=0.0000
feat-kd : total=400.9392, det=400.9318, feat=0.1496, kl=0.0000
grad mean sum -> student=22.634519, adapters=0.000064

KD preflight PASSED
READY_TO_TRAIN = True


In [17]:
imgs, _ = next(iter(train_loader))
print(imgs.min().item(), imgs.max().item(), imgs.mean().item(), imgs.std().item())

0.0 1.0 0.4485924243927002 0.219781294465065


In [18]:
# ================================================================
# CELL 14.5: CLEAN + OVERSAMPLE (RUN THIS BEFORE TRAINING!)
# ================================================================
import shutil
import random
from pathlib import Path
import yaml

print("="*70)
print("🔧 CLEANING + OVERSAMPLING")
print("="*70)

random.seed(42)

# Read current class count from data.yaml
with open(YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)
NC_NOW = data_cfg['nc']
CLASS_NAMES = data_cfg['names']
print(f"  Classes: {NC_NOW}")

train_img_dir = MERGED / 'train' / 'images'
train_lbl_dir = MERGED / 'train' / 'labels'

# ==========================================
# PART A: REMOVE FAKE BOXES
# ==========================================
print("\n🧹 PART A: Removing fake boxes...")

total_removed = 0
total_deleted = 0

for split in ['train', 'val', 'test']:
    img_dir = MERGED / split / 'images'
    lbl_dir = MERGED / split / 'labels'
    
    if not lbl_dir.exists():
        continue
    
    removed = 0
    deleted = 0
    
    for lbl_file in list(lbl_dir.glob('*.txt')):
        txt = lbl_file.read_text().strip()
        if not txt:
            continue
        
        new_lines = []
        found_fake = False
        
        for line in txt.splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            
            w = float(parts[3])
            h = float(parts[4])
            
            # Fake = covers more than 90% of image
            if w * h > 0.90 or (w > 0.95 and h > 0.95):
                removed += 1
                found_fake = True
            else:
                new_lines.append(line.strip())
        
        if found_fake:
            if len(new_lines) == 0:
                # No real boxes left → delete image + label
                lbl_file.unlink()
                for ext in ['.jpg', '.jpeg', '.JPG', '.JPEG', '.png', '.PNG']:
                    img_f = img_dir / f"{lbl_file.stem}{ext}"
                    if img_f.exists():
                        img_f.unlink()
                        deleted += 1
                        break
            else:
                # Keep image but remove fake boxes
                lbl_file.write_text('\n'.join(new_lines))
    
    print(f"  {split}: {removed} fake boxes removed, {deleted} images deleted")
    total_removed += removed
    total_deleted += deleted

if total_removed == 0:
    print("  ✅ No fake boxes found! Your data is clean.")
else:
    print(f"  ✅ Cleaned {total_removed} fake boxes, deleted {total_deleted} images")

# ==========================================
# PART B: COUNT CLASSES
# ==========================================
print(f"\n📊 PART B: Class distribution...")

class_to_images = {i: [] for i in range(NC_NOW)}

for lbl_file in sorted(train_lbl_dir.glob('*.txt')):
    txt = lbl_file.read_text().strip()
    if not txt:
        continue
    
    seen = set()
    for line in txt.splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = int(float(parts[0]))
            if 0 <= cls_id < NC_NOW and cls_id not in seen:
                class_to_images[cls_id].append(lbl_file.stem)
                seen.add(cls_id)

small_classes = []
for c in range(NC_NOW):
    count = len(class_to_images[c])
    name = CLASS_NAMES.get(c, f"class_{c}")
    status = "✅" if count >= 500 else "⚠️"
    if count < 500:
        small_classes.append(c)
    print(f"  {status} Class {c:2d} ({name}): {count}")

# ==========================================
# PART C: OVERSAMPLE SMALL CLASSES TO 500
# ==========================================
if len(small_classes) == 0:
    print(f"\n✅ All classes have 500+ images. No oversampling needed!")
else:
    print(f"\n📈 PART C: Oversampling {len(small_classes)} small classes to 500...")
    
    TARGET = 500
    total_added = 0
    
    for c in small_classes:
        current = len(class_to_images[c])
        
        if current == 0:
            print(f"  ⚠️ Class {c}: 0 images, cannot oversample!")
            continue
        
        needed = TARGET - current
        source_stems = class_to_images[c]
        
        added = 0
        for i in range(needed):
            src_stem = random.choice(source_stems)
            
            # Find source image
            src_img = None
            for ext in ['.jpg', '.jpeg', '.JPG', '.JPEG', '.png', '.PNG']:
                candidate = train_img_dir / f"{src_stem}{ext}"
                if candidate.exists():
                    src_img = candidate
                    break
            
            if src_img is None:
                continue
            
            src_lbl = train_lbl_dir / f"{src_stem}.txt"
            if not src_lbl.exists():
                continue
            
            # Copy with new name
            new_stem = f"os_{c}_{i:05d}"
            shutil.copy2(src_img, train_img_dir / f"{new_stem}{src_img.suffix}")
            shutil.copy2(src_lbl, train_lbl_dir / f"{new_stem}.txt")
            added += 1
            total_added += 1
        
        name = CLASS_NAMES.get(c, f"class_{c}")
        print(f"  Class {c} ({name}): {current} → {current + added} ✅")
    
    print(f"\n  ✅ Total added: {total_added} images")

# ==========================================
# PART D: FINAL VERIFICATION
# ==========================================
print(f"\n{'='*70}")
print("📊 FINAL DATASET:")

for split in ['train', 'val', 'test']:
    imgs = len(list((MERGED / split / 'images').glob('*')))
    lbls = len(list((MERGED / split / 'labels').glob('*.txt')))
    match = "✅" if imgs == lbls else "❌"
    print(f"  {match} {split}: {imgs} images, {lbls} labels")

print(f"\n✅ Data is clean and balanced!")
print(f"   Now run Cell 15 to train!")

🔧 CLEANING + OVERSAMPLING
  Classes: 20

🧹 PART A: Removing fake boxes...
  train: 105 fake boxes removed, 102 images deleted
  val: 7 fake boxes removed, 7 images deleted
  test: 20 fake boxes removed, 20 images deleted
  ✅ Cleaned 132 fake boxes, deleted 129 images

📊 PART B: Class distribution...
  ✅ Class  0 (Waterhemp): 3106
  ✅ Class  1 (MorningGlory): 2367
  ✅ Class  2 (Purslane): 1280
  ✅ Class  3 (SpottedSpurge): 1228
  ✅ Class  4 (Carpetweed): 1017
  ✅ Class  5 (Ragweed): 1046
  ✅ Class  6 (Eclipta): 1209
  ✅ Class  7 (PricklySida): 937
  ✅ Class  8 (PalmerAmaranth): 690
  ⚠️ Class  9 (Sicklepod): 434
  ⚠️ Class 10 (Goosegrass): 368
  ✅ Class 11 (Kena): 2682
  ✅ Class 12 (Lavhala): 4063
  ✅ Class 13 (Gajar Gavat): 4444
  ✅ Class 14 (Graceful Sandmart): 1082
  ✅ Class 15 (Sicklepod Mh): 817
  ✅ Class 16 (Harali): 1307
  ✅ Class 17 (OtherWeed): 592
  ✅ Class 18 (Lamber Quarter Plant): 1224
  ⚠️ Class 19 (Little Mallow): 0

📈 PART C: Oversampling 3 small classes to 500...
  Clas

In [19]:
# ================================================================
# CELL 14.6: REMOVE EMPTY CLASS 19 + FIX YAML
# ================================================================
import yaml
from pathlib import Path

print("="*70)
print("🔧 REMOVING EMPTY CLASS (Class 19: Little Mallow)")
print("="*70)

# Step 1: Read current yaml
with open(YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)

old_names = data_cfg['names']
old_nc = data_cfg['nc']
print(f"  Before: {old_nc} classes")

# Step 2: Find empty classes
empty_classes = [19]  # Class 19 has 0 images
print(f"  Removing: {[old_names[c] for c in empty_classes]}")

# Step 3: Build remap (old_id → new_id, skip empty)
new_names = {}
remap = {}
new_id = 0

for old_id in range(old_nc):
    if old_id in empty_classes:
        remap[old_id] = -1  # will be deleted
        continue
    remap[old_id] = new_id
    new_names[new_id] = old_names[old_id]
    new_id += 1

new_nc = len(new_names)
print(f"  After: {new_nc} classes")
print(f"  Remap: {remap}")

# Step 4: Remap ALL label files
for split in ['train', 'val', 'test']:
    lbl_dir = MERGED / split / 'labels'
    img_dir = MERGED / split / 'images'
    
    if not lbl_dir.exists():
        continue
    
    remapped = 0
    deleted = 0
    
    for lbl_file in list(lbl_dir.glob('*.txt')):
        txt = lbl_file.read_text().strip()
        if not txt:
            continue
        
        new_lines = []
        for line in txt.splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            
            old_cls = int(float(parts[0]))
            
            if old_cls not in remap or remap[old_cls] == -1:
                continue  # skip empty class
            
            new_cls = remap[old_cls]
            new_lines.append(f"{new_cls} {parts[1]} {parts[2]} {parts[3]} {parts[4]}")
        
        if len(new_lines) == 0:
            # No annotations left → delete
            lbl_file.unlink()
            for ext in ['.jpg', '.jpeg', '.JPG', '.JPEG', '.png', '.PNG']:
                img_f = img_dir / f"{lbl_file.stem}{ext}"
                if img_f.exists():
                    img_f.unlink()
                    break
            deleted += 1
        else:
            lbl_file.write_text('\n'.join(new_lines))
            remapped += 1
    
    print(f"  {split}: {remapped} remapped, {deleted} deleted")

# Step 5: Update data.yaml
data_cfg['nc'] = new_nc
data_cfg['names'] = new_names

with open(YAML_PATH, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\n✅ Updated {YAML_PATH}")
print(f"   {new_nc} classes:")
for i, name in new_names.items():
    print(f"   {i}: {name}")

# Step 6: Final count
print(f"\n📊 Final dataset:")
for split in ['train', 'val', 'test']:
    imgs = len(list((MERGED / split / 'images').glob('*')))
    lbls = len(list((MERGED / split / 'labels').glob('*.txt')))
    match = "✅" if imgs == lbls else "❌"
    print(f"  {match} {split}: {imgs} images, {lbls} labels")

print(f"\n✅ Ready to train with {new_nc} classes!")

🔧 REMOVING EMPTY CLASS (Class 19: Little Mallow)
  Before: 20 classes
  Removing: ['Little Mallow']
  After: 19 classes
  Remap: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: -1}
  train: 17662 remapped, 0 deleted
  val: 2000 remapped, 0 deleted
  test: 3037 remapped, 0 deleted

✅ Updated /kaggle/working/merged/data.yaml
   19 classes:
   0: Waterhemp
   1: MorningGlory
   2: Purslane
   3: SpottedSpurge
   4: Carpetweed
   5: Ragweed
   6: Eclipta
   7: PricklySida
   8: PalmerAmaranth
   9: Sicklepod
   10: Goosegrass
   11: Kena
   12: Lavhala
   13: Gajar Gavat
   14: Graceful Sandmart
   15: Sicklepod Mh
   16: Harali
   17: OtherWeed
   18: Lamber Quarter Plant

📊 Final dataset:
  ✅ train: 17662 images, 17662 labels
  ✅ val: 2000 images, 2000 labels
  ✅ test: 3037 images, 3037 labels

✅ Ready to train with 19 classes!


In [20]:
# ================================================================
# CELL 15: TRAIN YOLOv8s
# ================================================================
from ultralytics import YOLO
import shutil

print("="*70)
print("🚀 TRAINING YOLOv8s - 19 classes")
print("="*70)

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(YAML_PATH),
    
    epochs=30,
    patience=40,
    
    imgsz=640,
    batch=16,
    
    cache='disk',
    workers=4,
    amp=True,
    
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    optimizer='AdamW',
    weight_decay=0.0005,
    
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    flipud=0.0,
    
    cos_lr=True,
    close_mosaic=15,
    
    box=7.5,
    cls=0.5,
    dfl=1.5,
    
    device=0,
    project=str(OUTPUT),
    name='yolo_train',
    exist_ok=True,
    verbose=True,
    plots=True,
    save=True,
    save_period=20,
    val=True,
    seed=42,
    deterministic=False,
)

best = OUTPUT / 'yolo_train' / 'weights' / 'best.pt'
if best.exists():
    shutil.copy2(best, OUTPUT / 'best_student_final.pt')

try:
    print(f"\n✅ Best mAP50:    {float(results.box.map50):.4f}")
    print(f"✅ Best mAP50-95: {float(results.box.map):.4f}")
except:
    print("Check results folder")

🚀 TRAINING YOLOv8s - 19 classes
New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/merged/data.yaml, degrees=10.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, mult